# MedTrace Clinical VLM Training Lab

A hackathon-ready pipeline built for MedTrace AI: QLoRA fine-tuning of `google/medgemma-4b-it`, experiment tracking with Weights & Biases, held-out evaluation, and gated model publishing to Hugging Face.

**Runtime:** A100 GPU &nbsp;|&nbsp; **Model:** MedGemma 4B &nbsp;|&nbsp; **Method:** 4-bit LoRA &nbsp;|&nbsp; **Tracking:** W&B

In [ ]:
%pip install -q -U "transformers>=4.50,<6" "trl>=0.28,<2" "peft>=0.15,<1" "accelerate>=1.7,<2" "bitsandbytes>=0.45,<1" "datasets>=3.6,<5" "evaluate>=0.4,<1" "wandb>=0.19,<1" "rouge-score>=0.1.2,<1" "hf_xet>=1,<2"

## 1. Load the MedTrace training engine

Every run records the exact MedTrace commit for reproducibility.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/pramodthe/MedTrace-AI.git"
REPO_REF = "main"  # Prefer a commit SHA or release tag for a real run.
REPO_DIR = Path("/content/MedTrace-AI")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
print("Training code commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())

## 2. Connect Hugging Face and W&B

Add `HF_TOKEN` and `WANDB_API_KEY` in the Colab Secrets panel, then enable notebook access for both.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
assert os.environ["HF_TOKEN"], "Add HF_TOKEN to Colab Secrets and enable notebook access."
assert os.environ["WANDB_API_KEY"], "Add WANDB_API_KEY to Colab Secrets and enable notebook access."
print("Secrets loaded (values hidden).")

## 3. Configure the experiment

Choose either `DATASET_ID` for a Hugging Face DatasetDict or `DATASET_PATH` for a Drive/local dataset. Required columns are `image`, `prompt`, and `response`. A `patient_id` or study-level group column enables a clean group-based validation split.

For report generation, `response` is JSON with `summary`, `findings`, `impression`, `recommendation`, and numeric `confidence`. Set `PUSH_TO_HUB=True` to publish a run that passes the evaluation gates.

In [ ]:
# Data source: set exactly one.
DATASET_ID = ""          # Example: "your-org/deidentified-radiology-reports"
DATASET_PATH = ""        # Example: "/content/drive/MyDrive/medtrace/dataset"
DATASET_CONFIG = ""
TRAIN_SPLIT = "train"
EVAL_SPLIT = "validation"
IMAGE_COLUMN = "image"
PROMPT_COLUMN = "prompt"
RESPONSE_COLUMN = "response"
GROUP_COLUMN = "patient_id"
TASK_MODE = "report"  # report | classification | free_text

# Tracking and publication.
WANDB_PROJECT = "medtrace-medgemma"
WANDB_ENTITY = ""
RUN_NAME = "medgemma-4b-medtrace-qlora-v1"
HUB_MODEL_ID = ""      # Example: "your-user/medtrace-medgemma-4b-radiology-lora"
HUB_PRIVATE = True
PUSH_TO_HUB = False
LOG_EVAL_SAMPLES_TO_WANDB = False

# Fast A100 hackathon run. Set either sample cap to None for the full dataset.
MAX_TRAIN_SAMPLES = 1000  # None for all rows
MAX_EVAL_SAMPLES = 200    # None for all rows
GENERATION_EVAL_SAMPLES = 32
EPOCHS = 1
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LORA_RANK = 16
MIN_ROUGE_L_DELTA = 0.0
MIN_JSON_VALID_RATE = 0.95

assert bool(DATASET_ID) ^ bool(DATASET_PATH), "Set exactly one of DATASET_ID or DATASET_PATH."
if PUSH_TO_HUB:
    assert HUB_MODEL_ID and "/" in HUB_MODEL_ID, "Set a complete HUB_MODEL_ID before publishing."

## 4. Train, evaluate, and publish

This stage compares the base and tuned models, saves the adapter and run artifacts, logs metrics to W&B, and publishes successful runs to Hugging Face when enabled.

In [ ]:
import subprocess

OUTPUT_DIR = "/content/medtrace-medgemma-output"
command = [
    "python", str(REPO_DIR / "scripts/finetune_medgemma_colab.py"),
    "--model-id", "google/medgemma-4b-it",
    "--train-split", TRAIN_SPLIT,
    "--eval-split", EVAL_SPLIT,
    "--image-column", IMAGE_COLUMN,
    "--prompt-column", PROMPT_COLUMN,
    "--response-column", RESPONSE_COLUMN,
    "--group-column", GROUP_COLUMN,
    "--task-mode", TASK_MODE,
    "--output-dir", OUTPUT_DIR,
    "--wandb-project", WANDB_PROJECT,
    "--run-name", RUN_NAME,
    "--generation-eval-samples", str(GENERATION_EVAL_SAMPLES),
    "--epochs", str(EPOCHS),
    "--learning-rate", str(LEARNING_RATE),
    "--train-batch-size", str(TRAIN_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION_STEPS),
    "--lora-rank", str(LORA_RANK),
    "--min-rouge-l-delta", str(MIN_ROUGE_L_DELTA),
    "--min-json-valid-rate", str(MIN_JSON_VALID_RATE),
]
command += ["--dataset-id", DATASET_ID] if DATASET_ID else ["--dataset-path", DATASET_PATH]
if DATASET_CONFIG:
    command += ["--dataset-config", DATASET_CONFIG]
if WANDB_ENTITY:
    command += ["--wandb-entity", WANDB_ENTITY]
if MAX_TRAIN_SAMPLES is not None:
    command += ["--max-train-samples", str(MAX_TRAIN_SAMPLES)]
if MAX_EVAL_SAMPLES is not None:
    command += ["--max-eval-samples", str(MAX_EVAL_SAMPLES)]
if LOG_EVAL_SAMPLES_TO_WANDB:
    command.append("--log-eval-samples-to-wandb")
if PUSH_TO_HUB:
    command += ["--push-to-hub", "--hub-model-id", HUB_MODEL_ID]
    command.append("--hub-private" if HUB_PRIVATE else "--no-hub-private")

subprocess.run(command, check=True)

## 5. View the results

Open the W&B run to compare loss curves and evaluation metrics. The local prediction file supports the MedTrace clinician-review loop.

In [ ]:
import json
from pathlib import Path

metrics = json.loads((Path(OUTPUT_DIR) / "eval_metrics.json").read_text())
display(metrics)
print("Local clinician-review file:", Path(OUTPUT_DIR) / "eval_predictions.jsonl")
print("Hugging Face publication requested:", PUSH_TO_HUB)